In [0]:
%pip install --quiet -U databricks-sdk==0.80.0
dbutils.library.restartPython()

In [0]:
%run ./Classroom-Setup-Common

In [0]:
DA.display_config_values([('Course Catalog',DA.catalog_name),('Your Schema',DA.schema_name)])

In [0]:
import os

curr_path = os.getcwd()
print(f"Current Path: {curr_path}")

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import sql

w = WorkspaceClient()

def create_query_file(catalog, schema, table_name):

    # Retrieve available data sources (SQL warehouses)
    srcs = w.data_sources.list()
    query_name = "1.2 - Creating sales table - SQL Query"

    # Check if a query with the same name already exists
    query_id = validate_job_query_file_exists(query_name)

    # If the query exists, delete it to avoid duplicates
    if query_id != -1:
        w.queries.delete(id=query_id)
        print(f'Query Deleted id {query_id}')

    # Create a new query to generate the sales table using CTAS
    query = w.queries.create(
        query=sql.CreateQueryRequestQuery(
            display_name=query_name,
            warehouse_id=srcs[0].warehouse_id,  # Use the first available warehouse
            description="CTAS for Sales Table for Demo 1",
            parent_path = f"{curr_path}/Task Files/Lesson 1 Files",
            query_text=f"""
CREATE OR REPLACE TABLE {catalog}.{schema}.{table_name} AS SELECT * FROM dbacademy_retail.v01.sales
"""
            )
    )
    print(f'Query Created id {query.id}')

    return query

def validate_job_query_file_exists(query_name):
    # List all queries and check if one matches the given name
    query_list = w.queries.list()
    for query in query_list:
        if query.display_name == query_name:
            return query.id
    return -1  # Return -1 if no matching query is found

In [0]:
query = create_query_file(f"{DA.catalog_name}",f"{DA.schema_name}", "sales_bronze")